# Exercise 7: From the log to the lake

Every run of your job does three things: **read** the new records of the topic `logs`,
**write** them into the lake (Parquet files in `data/lake/`), **remember** where it stopped.

This notebook gives you the building blocks (`tools/lakelib.py`). You write one function,
`run_once()`. Save your final version of this notebook into your `solution/` folder.

In [1]:
import sys; sys.path.append("/home/jovyan/tools")
from lakelib import *

## 1. The example: a job that copies everything

It reads the topic from the beginning, every time. Run the next two cells and look at the
checker's last line.

In [2]:
def run_once():
    new = read_new(None)          # None = from the beginning of the topic
    write_files(new, LAKE)        # straight into the lake

reset()                           # empty lake
run_once()
check()

reset: lake, staging and state deleted
read_new: 2,700,000 new records
write_files: 2,700,000 records -> /home/jovyan/data/lake/logs/run-20261119-143532-515570

Lake check — topic 'logs' (4 partitions)   2026-11-19 14:37:44
lake: read_parquet('/home/jovyan/data/lake/logs/**/*.parquet')

partition          topic offsets     in lake  duplicates    missing  not loaded
-------------------------------------------------------------------------------
        0           [0, 706,715)     706,715           0          0           0
        1           [0, 663,115)     663,115           0          0           0
        2           [0, 628,995)     628,995           0          0           0
        3           [0, 701,175)     701,175           0          0           0
-------------------------------------------------------------------------------
    total      2,700,000 records   2,700,000           0          0           0

files under /home/jovyan/data/lake: 4 data files, 123.5 MB, 30.86 MB pe

{'topic': 2700000,
 'lake': 2700000,
 'dup': 0,
 'miss': 0,
 'lag': 0,
 'files': 4,
 'mb': 123.45488,
 'ok': True,
 'problems': []}

In [3]:
run_once()                        # the next run
check()

read_new: 2,700,000 new records
write_files: 2,700,000 records -> /home/jovyan/data/lake/logs/run-20261119-143744-06d1f5

Lake check — topic 'logs' (4 partitions)   2026-11-19 14:39:19
lake: read_parquet('/home/jovyan/data/lake/logs/**/*.parquet')

partition          topic offsets     in lake  duplicates    missing  not loaded
-------------------------------------------------------------------------------
        0           [0, 706,715)   1,413,430     706,715          0           0
        1           [0, 663,115)   1,326,230     663,115          0           0
        2           [0, 628,995)   1,257,990     628,995          0           0
        3           [0, 701,175)   1,402,350     701,175          0           0
-------------------------------------------------------------------------------
    total      2,700,000 records   5,400,000   2,700,000          0           0

files under /home/jovyan/data/lake: 8 data files, 246.9 MB, 30.86 MB per file on average
check took 1.0 s

Exa

{'topic': 2700000,
 'lake': 5400000,
 'dup': 2700000,
 'miss': 0,
 'lag': 0,
 'files': 8,
 'mb': 246.90976,
 'ok': False,
 'problems': ['2,700,000 duplicates']}

## 2. The building blocks

| Block | What it does |
|---|---|
| `load_state()` | where the last run stopped, from a small state file (`None` if there is none) |
| `save_state(new.end)` | stores where this run stopped in the state file |
| `lake_position()` | where the last run stopped, read from the lake itself (`None` if the lake is empty) |
| `read_new(start)` | the records after `start`, up to the end of the topic *now* (`new.end`) |
| `write_files(new, folder)` | writes them as Parquet into a new run folder inside `LAKE` or `STAGING` |
| `publish(run)` | moves a finished run folder from `STAGING` into the lake: one rename, a single step |

`LAKE` is what readers see; `STAGING` is a folder next to it that nobody reads.
`new.df.show()` shows the records of a run as a Spark DataFrame.

## 3. Your architecture

Write `run_once()` for the architecture you chose (A, B or C, see the README).
The order of the calls *is* your architecture.

In [7]:
def run_once():
    # 1. where did the last run stop?
    ## lake_position() -> where the last run stopped, read from the lake itself (None if the lake is empty)
    start = lake_position()
    
    # 2. read what is new
    new = read_new(start)
    
    # 3. write it
    ## writes them as Parquet into a new run folder inside STAGING
    run = write_files(new, STAGING)
    ## moves a finished run folder from STAGING into the lake
    publish(run)
    
    # 4. remember where you stopped (if your architecture needs it)
    ## I don't need it, because option B derives the position 
    ## of the job from the data

## 4. Check it

`check_all` empties the lake, then runs your job in five situations: a first run, new records,
nothing new, a run that stops **while** writing, a run that stops **after** writing.
It takes 2-4 minutes.

Architecture C: readers use a view, so check through it: `check_all(run_once, lake_sql=DEDUP_SQL)`.

In [8]:
check_all(run_once)

reset: lake, staging and state deleted

situation                                 in lake  duplicates  missing  not loaded    run  check
-----------------------------------------------------------------------------------------------
1 first run                             3,000,000           0        0           0 125.3s  PASS
2 new records arrive, run again         3,100,000           0        0           0  12.2s  PASS
3 nothing new, run again                3,100,000           0        0           0   0.9s  PASS
4 run stops WHILE writing: right after  3,100,000           0        0     100,000         PASS
  ... then the next run                 3,200,000           0        0           0  13.1s  PASS
5 run stops AFTER writing: right after  3,200,000           0        0     100,000         PASS
  ... then the next run                 3,300,000           0        0           0  13.5s  PASS
-----------------------------------------------------------------------------------------------

[('1 first run',
  {'topic': 3000000,
   'lake': 3000000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 4,
   'mb': 128.618322,
   'ok': True,
   'problems': []},
  125.33599019050598),
 ('2 new records arrive, run again',
  {'topic': 3100000,
   'lake': 3100000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 8,
   'mb': 129.72304,
   'ok': True,
   'problems': []},
  12.164722204208374),
 ('3 nothing new, run again',
  {'topic': 3100000,
   'lake': 3100000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 8,
   'mb': 129.72304,
   'ok': True,
   'problems': []},
  0.8963406085968018),
 ('4 run stops WHILE writing: right after',
  {'topic': 3200000,
   'lake': 3100000,
   'dup': 0,
   'miss': 0,
   'lag': 100000,
   'files': 8,
   'mb': 129.72304,
   'ok': True,
   'problems': []},
  None),
 ('  ... then the next run',
  {'topic': 3200000,
   'lake': 3200000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 12,
   'mb': 130.878339,
   'ok': True,
   'problems': []},
  